In [34]:
import sys
sys.path.append("../") # Go up one level to project root 
#from src_test import drugbank_parse_fast
import pandas as pd
import zipfile
import xml.etree.ElementTree as ET
import random
import copy
import os

In [36]:
# Set your XML file path, 
## You have to add the zip file to this path since git repo doesn't track this large file
xml_path = "../data/raw/drugbank_all_full_database_V5.1.14.xml.zip"
ns = "{http://www.drugbank.ca}"

In [26]:
def parse_drug_element(drug_element): 
    """Recursively extract all data from a drug element into a flat dict."""
    drug_data = {}
    
    def extract_all(element, prefix=""):
        tag = element.tag.replace(ns, "")
        full_key = f"{prefix}/{tag}" if prefix else tag
        
        # get text content
        if element.text and element.text.strip():
            drug_data[full_key]=element.text.strip()
            
        # Get attributes
        for attr_name, attr_val in element.attrib.items():
            drug_data[f"{full_key}[@{attr_name}]"]= attr_val
            
        # Recurse into children
        for child in element:
            extract_all(child, full_key)

    extract_all(drug_element)
    return drug_data


In [35]:
def get_random_drugs(xml_zip_file, n=5, seed=None):
    """ Stream through zipped DrugBank XML and reservior-sample n drugs."""

    # Set seed: used provided seed or generate one from OS entropy
    if seed is None: 
        seed = int.from_bytes(os.urandom(4), 'big')
    random.seed(seed)
    print(f"Save the Random seed: {seed} used if want to reproduce the same sample set")
    
    drug_tag = f"{ns}drug"
    primary_id_tag = f"{ns}drugbank-id[@primary='true']"
    reservoir = []
    count = 0

    with zipfile.ZipFile(xml_zip_file,'r') as z:
        xml_filename = [f for f in z.namelist() if f.endswith('.xml')][0]
        print(f"Streaming: {xml_filename}")

        with z.open(xml_filename) as xml_file:
            for event, elem in ET.iterparse(xml_file, events = ("end",)):

                # Only process top-level drug elements that have primary drugbank-id
                if elem.tag == drug_tag and elem.find(primary_id_tag) is not None:
                    count += 1

                    # Reservoir sampling
                    if len(reservoir) < n:
                        reservoir.append(copy.deepcopy(elem))
                    else:
                        j = random.randint(0, count -1)
                        if j < n:
                            reservoir[j] = copy.deepcopy(elem)
 
                    elem.clear() # free memory

                    if count%1000 == 0:
                        print(f" Scanned {count} drug...")
                        
    print(f"\n Total drugs scanned: {count} | Randomly selected: {len(reservoir)}")
    return reservoir

In [37]:
def drugs_to_dataframe(xml_zip_file, n=5):
    """Sample n drugs and save all their data into a DataFrame."""

    drugs = get_random_drugs(xml_zip_file, n)

    # Parse each drug into a dict, collect all dicts into a list
    records = []
    for drug_element in drugs:
        drug_data = parse_drug_element(drug_element)
        records.append(drug_data)

    # pd.DataFrame from a list of dicts automatically: 
    # - creates a column for every key seen across all drugs
    # - fills NaN where a drug doesn't have that field
    df = pd.DataFrame(records)

    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns ({len(df.columns)}):")
    for col in df.columns:
        non_null = df[col].notna().sum()
        print(f" {col}: {non_null}/{len(df)} drugs have data")

    return df 

In [38]:
df = drugs_to_dataframe(xml_path, n=1000)

Save the Random seed: 2688059193 used if want to reproduce the same sample set
Streaming: full database.xml
 Scanned 1000 drug...
 Scanned 2000 drug...
 Scanned 3000 drug...
 Scanned 4000 drug...
 Scanned 5000 drug...
 Scanned 6000 drug...
 Scanned 7000 drug...
 Scanned 8000 drug...
 Scanned 9000 drug...
 Scanned 10000 drug...
 Scanned 11000 drug...
 Scanned 12000 drug...
 Scanned 13000 drug...
 Scanned 14000 drug...
 Scanned 15000 drug...
 Scanned 16000 drug...
 Scanned 17000 drug...
 Scanned 18000 drug...
 Scanned 19000 drug...

 Total drugs scanned: 19830 | Randomly selected: 1000

DataFrame shape: (1000, 318)
Columns (318):
 drug[@type]: 1000/1000 drugs have data
 drug[@created]: 1000/1000 drugs have data
 drug[@updated]: 1000/1000 drugs have data
 drug/drugbank-id: 1000/1000 drugs have data
 drug/drugbank-id[@primary]: 1000/1000 drugs have data
 drug/name: 1000/1000 drugs have data
 drug/description: 631/1000 drugs have data
 drug/cas-number: 657/1000 drugs have data
 drug/unii: 7

In [39]:
# Save to csv 
df.to_csv("../data/sample/randomly_sampled_1000_drugs.csv", index=False)
print("\nSaved to ddi_prediction/data/sample")


Saved to ddi_prediction/data/sample


In [49]:
# Transpose and disply top drugs as columns and their attributes as rows
pd.set_option('display.max_rows',None)
pd.set_option('display.max_colwidth',50)
df.head(5).T

,0,1,2,3,4
drug[@type],small molecule,biotech,biotech,biotech,small molecule
drug[@created],2020-12-15,2018-07-05,2019-05-20,2005-06-13,2007-11-18
drug[@updated],2025-04-23,2025-04-02,2025-04-23,2025-11-04,2025-04-02
drug/drugbank-id,DB16048,DB14397,DB15081,BIOD00084,DB05443
drug/drugbank-id[@primary],true,true,true,true,true
drug/name,BTRX-246040,Influenza B virus B/Massachusetts/2/2012 BX-51...,Melanoma antigen recognized by T-cells 1,Denileukin diftitox,ADL 10-0101
drug/description,BTRX-246040 is under investigation in clinical...,A seasonally-specific component of the influen...,Melanoma antigen recognized by T-cells 1 is un...,Denileukin diftitox is an IL2-receptor-directe...,ADL 10-0101 is a peripheral kappa opioid agoni...
drug/cas-number,1307245-86-8,NaN,NaN,173146-27-5,NaN
drug/unii,4I67US2V8Q,N784VJP7ZF,C1357NAP5J,25E79B5CTM,NaN
drug/average-mass,480.96,NaN,NaN,NaN,NaN
